# Predicting PM2.5 Air Quality using a combination of satellite, low cost and reference sensor data


## Notebook overview
This notebook trains and evaluates a baseline PM2.5 model using satellite, reference, and low-cost sensor data. It walks through loading the silver parquet data, quick EDA, feature engineering, model training/validation, and SHAP-based interpretation. Update the configuration cell below before running.

In [ ]:
import datetime
import os
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import xgboost as xgb
import shap
import plotly.express as px
import matplotlib.pyplot as plt
import polars_h3 as plh3
from IPython.display import display
from xgboost import XGBRegressor

plt.style.use("seaborn-v0_8")
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(20)
from plotly.subplots import make_subplots
import math


In [ ]:
# Helpers

def gridify(figs, cols=3, height_per_row=320, width_per_col=380, shared_x=False, shared_y=False, titles=None):
    n=len(figs); rows=math.ceil(n/cols)
    fig=make_subplots(rows=rows, cols=cols, shared_xaxes=shared_x, shared_yaxes=shared_y,
                      #subplot_titles=titles or [(f.layout.title.text or f"Plot {i+1}") for i,f in enumerate(figs)],
                      horizontal_spacing=0.06, vertical_spacing=0.08)
    for i,f in enumerate(figs):
        r=i//cols+1; c=i%cols+1
        for tr in f.data: fig.add_trace(tr, row=r, col=c)
        xt=getattr(f.layout.xaxis.title, "text", None); yt=getattr(f.layout.yaxis.title, "text", None)
        if xt: fig.update_xaxes(title_text=xt, row=r, col=c)
        if yt: fig.update_yaxes(title_text=yt, row=r, col=c)
    fig.update_layout(height=rows*height_per_row, width=cols*width_per_col, showlegend=False,
                      margin=dict(t=50,r=10,b=10,l=10))
    return fig

## Experiment settings

In [ ]:
EXPERIMENT = {
    "log_target": True,
    "log_transform": "log1p",  # choose between 'log1p' and 'log'
}

In [ ]:
from glob import glob


def _find_silver_glob_candidates(search_roots):
    candidates = []
    discovery_patterns = [
        "**/silver/historical/*.parquet",
        "**/silver/*/historical/*.parquet",
        "**/historical/*.parquet",
    ]
    for root in search_roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        for pattern in discovery_patterns:
            for parquet_file in root_path.glob(pattern):
                if not parquet_file.is_file():
                    continue
                parent_lower = str(parquet_file.parent).lower()
                if "silver" not in parent_lower or "historical" not in parent_lower:
                    continue
                candidates.append(str(parquet_file.parent / "*.parquet"))
    return sorted(set(candidates))


def resolve_silver_path(user_glob=None):
    default_candidates = [
        user_glob,
        os.getenv("SILVER_PATH"),
        "/app/data/silver/historical/*.parquet",
        str(Path.cwd() / "data" / "silver" / "historical" / "*.parquet"),
        str(Path.cwd().parent / "data" / "silver" / "historical" / "*.parquet"),
    ]
    checked = []
    for candidate in default_candidates:
        if not candidate:
            continue
        checked.append(candidate)
        if glob(candidate):
            return candidate

    discovered = _find_silver_glob_candidates(["/app", str(Path.cwd()), str(Path.home())])
    if discovered:
        print("Could not find silver parquet in default locations.")
        print("Discovered candidate parquet folders:")
        for i, candidate in enumerate(discovered, 1):
            print(f"  {i}. {candidate}")
        selection = input(
            "Select a number from the list above, or paste a custom parquet glob path: "
        ).strip()
        if selection.isdigit() and 1 <= int(selection) <= len(discovered):
            return discovered[int(selection) - 1]
        if selection:
            if glob(selection):
                return selection
            raise FileNotFoundError(f"No parquet files matched custom path: {selection}")

    raise FileNotFoundError(
        "No silver parquet files found. Checked: "
        + ", ".join(checked)
        + ". Set SILVER_PATH or pass user_glob to resolve_silver_path()."
    )


silver_path = resolve_silver_path()
prepared_lazy = (
    pl.scan_parquet(silver_path, cast_options=pl.ScanCastOptions(float_cast="upcast"))
    .filter(pl.col("pm25_value").is_not_null())
    .sort(["cell", "date"])
)
print(f"Loading silver data from {silver_path}")


## Exploratory data analysis

In [ ]:
pm25_values = prepared_lazy.select("pm25_value").collect()
px.histogram(pm25_values)

Long tail PM2.5 distribution with occasional 350+ outliers; consider capping or winsorizing those extremes.

### Quick feature/target relationship check

In [ ]:
# Polars display all columns by default
pl.Config.set_fmt_str_lengths(30)  # limit string length display to 30 characters


In [ ]:
example = prepared_lazy.head().collect()

In [ ]:
example.columns

In [ ]:
quick_comparison_cols = ["aod_1day_interpolated", 
                                       "fire_hotspot_strength",
                                       "dewpoint_2m",
                                       "pm25_value"]
quick_comparison = prepared_lazy.select(["aod_1day_interpolated", 
                                       "fire_hotspot_strength",
                                       "dewpoint_2m",
                                       "pm25_value"]).collect().sample(10000)
figs = list()
for col in quick_comparison_cols:
    if col != "pm25_value":
        figs.append(px.scatter(quick_comparison, x=col, y="pm25_value", trendline="ols", trendline_color_override="red"))
grid = gridify(figs, cols=4)

In [ ]:
grid.show()

## Feature engineering / data prep
Using yesterday's mean PM2.5 from nearby cells as an additional feature.

In [ ]:
dataset_lazy = prepared_lazy
print(f"Feature columns available: {len(dataset_lazy.collect_schema().names())}")

In [ ]:
# For a baseline model, we'll train on 2022/2023/2024, and test on 2025 data (Which includes no low cost sensor data from Laos - They weren't deployed until March 2025)
# # Train/validation split by date

split_date = datetime.datetime(2025, 1, 1)

train_lazy = dataset_lazy.filter(pl.col("date") < split_date)
valid_lazy = dataset_lazy.filter(pl.col("date") >= split_date)


In [ ]:
valid_lazy = valid_lazy.filter(pl.col("pm25_value") < 250)

In [ ]:
# Collect data and prepare matrices for XGBoost

# Prune unused columns before collect to save memory
id_cols = {
    "cell", "date", "ISO3", "pm25_source", "day", "year", "parent_h3_04", "current_day_pm25",
}
landcover_cols = {
    "elevation", "trees", "grass", "shrub_and_scrub", "crops",
    "bare", "snow_and_ice", "flooded_vegetation", "built", "water",
}
drop_cols = id_cols | landcover_cols

schema_cols = dataset_lazy.collect_schema().names()
feature_cols = [c for c in schema_cols if c not in drop_cols and c != "pm25_value"]

# Collect only the needed columns
train_model = train_lazy.select(feature_cols + ["pm25_value"]).collect()
valid_model = valid_lazy.select(feature_cols + ["pm25_value"]).collect()




In [ ]:
sample_weights = train_model.select(
    pl.when(pl.col("pm25_value") <= 12).then(1.0)
    .when(pl.col("pm25_value") <= 35.4).then(1.5)
    .when(pl.col("pm25_value") <= 55.4).then(2.0)
    .when(pl.col("pm25_value") <= 150.4).then(2.5)
    .otherwise(6.0)
    .alias("sample_weight")
).to_series().to_numpy()

In [ ]:

feature_cols = [c for c in train_model.columns if c != "pm25_value"]

X_train_pd = train_model.select(feature_cols).to_pandas().astype(np.float32)
X_valid_pd = valid_model.select(feature_cols).to_pandas().astype(np.float32)

y_train_real = train_model["pm25_value"].to_numpy()
y_valid_real = valid_model["pm25_value"].to_numpy()

if EXPERIMENT["log_target"]:
    if EXPERIMENT.get("log_transform", "log1p") == "log":
        y_train = np.log(np.clip(y_train_real, 1e-6, None))
        y_valid = np.log(np.clip(y_valid_real, 1e-6, None))
        inverse_transform = np.exp
        label_description = "log(pm25_value)"
    else:
        y_train = np.log1p(np.clip(y_train_real, 0, None))
        y_valid = np.log1p(np.clip(y_valid_real, 0, None))
        inverse_transform = np.expm1
        label_description = "log1p(pm25_value)"
else:
    y_train = y_train_real
    y_valid = y_valid_real
    inverse_transform = lambda arr: arr
    label_description = "pm25_value"

print(f"Prepared {X_train_pd.shape[0]:,} training rows with {X_train_pd.shape[1]} features.")
print(f"Prepared {X_valid_pd.shape[0]:,} validation rows.")
print(f"Target fed to model: {label_description}")


In [ ]:
model = XGBRegressor(random_state=42, objective='reg:squarederror')

In [ ]:
model.fit(X_train_pd, y_train, sample_weight=sample_weights)

In [ ]:
# Evaluate on the validation set in the original PM2.5 space
preds_transformed = model.predict(X_valid_pd)
preds = inverse_transform(preds_transformed)

valid_full = valid_lazy.collect().with_columns(
    pm25_predicted=pl.Series(preds),
    pm25_value_real=pl.Series(y_valid_real),
    residual=pl.Series(preds - y_valid_real),
)

In [ ]:
true = y_valid_real
residuals = preds - true

mse = np.mean(residuals ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(residuals))
medae = np.median(np.abs(residuals))

ss_res = np.sum((true - preds) ** 2)
ss_tot = np.sum((true - np.mean(true)) ** 2)
r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")

mask = np.abs(true) > 1e-8
mape = (np.mean(np.abs((preds[mask] - true[mask]) / true[mask])) * 100) if mask.any() else float("nan")

print(f"Validation samples: {len(true):,}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Median AE: {medae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"R^2: {r2:.4f}")

if EXPERIMENT["log_target"]:
    log_mae = np.mean(np.abs(preds_transformed - y_valid))
    print(f"MAE (log space): {log_mae:.4f}")

validation_results = valid_full.with_columns([
    pl.Series("pm25_predicted", preds),
    pl.Series("pm25_predicted_transformed", preds_transformed),
    pl.Series("residual", residuals),
])

In [ ]:
valid_full.select(["pm25_value"]).describe()

In [ ]:
# Scatter plot of predictions vs observations
scatter_df = validation_results.select(["pm25_value", "pm25_predicted"]).to_pandas()


In [ ]:
fig = px.scatter(
    scatter_df,
    x="pm25_value",
    y="pm25_predicted",
    trendline="ols",
    trendline_color_override="red",
    title="Validation predictions vs. observed PM2.5",
    labels={"pm25_value": "Observed PM2.5", "pm25_predicted": "Predicted PM2.5"},
)
fig.update_layout(height=500)
fig

In [ ]:
# Residual distribution to spot any skew or bias
residual_df = validation_results.select(["residual"]).to_pandas()
fig = px.histogram(
    residual_df,
    x="residual",
    nbins=60,
    title="Residual distribution (predicted - observed)",
)
fig.update_layout(height=400)
fig

Residuals cluster around zero with a slight left lean.

In [ ]:
daily_eval = (
    validation_results
    .select(["date", "pm25_value", "pm25_predicted"])
    .group_by("date")
    .mean()
    .sort("date")
)

fig = px.line(
    daily_eval.to_pandas(),
    x="date",
    y=["pm25_value", "pm25_predicted"],
    title="Daily average PM2.5: observed vs predicted",
    labels={"value": "PM2.5"},
)
fig.update_layout(height=450, legend_title_text="Series")
fig

In [ ]:
# Feature importance via SHAP for interpretability
background_size = min(1000, len(X_train_pd))
if len(X_train_pd) > background_size:
    background_pd = X_train_pd.sample(n=background_size, random_state=42)
else:
    background_pd = X_train_pd

explainer = shap.explainers.TreeExplainer(model, data=background_pd)

evaluation_size = min(2000, len(X_valid_pd))
if len(X_valid_pd) > evaluation_size:
    evaluation_pd = X_valid_pd.sample(n=evaluation_size, random_state=42)
else:
    evaluation_pd = X_valid_pd

shap_values = explainer(evaluation_pd)
print(f"SHAP background size: {background_pd.shape}")
print(f"SHAP evaluation size: {evaluation_pd.shape}")

In [ ]:
validation_pd = validation_results.to_pandas()

In [ ]:
# Plot XGBoost feature importance for `model`
top_n = 30
feat_imp = pd.Series(model.feature_importances_, index=X_train_pd.columns).sort_values()
plot_imp = feat_imp.tail(min(top_n, len(feat_imp)))


In [ ]:
top = pd.DataFrame(plot_imp).reset_index().rename(columns={"index": 'feature', 0: 'importance'})

In [ ]:
top['Feature Type'] = top['feature'].apply(lambda x: x.split('_')[0])

In [ ]:
top = top.groupby('Feature Type').sum().drop("feature", axis=1).sort_values('importance')

In [ ]:
mapping = {
    "yesterday": "Nearby Ground Sensor",
    "fire": "Fire Hotspot Intensity",
    "wind": "Wind speed / direction",
    "aod": "Aerosol Optical Depth",
    "dewpoint": "Dewpoint",
    "temperature": "Temperature",
    "worldpop": "Population Density",
    "month": "Month"
}

top = top.rename(index=mapping)

In [ ]:
px.bar(top, orientation="h", title="XGBoost Feature Importance by Feature Type")

In [ ]:
top = feat_imp

In [ ]:
newtop = top.groupby('Feature Type').sum()

In [ ]:
newtop

In [ ]:
evaluation_summary = pd.DataFrame({
    "shap_row": np.arange(len(evaluation_pd)),
    "orig_index": evaluation_pd.index,
})

In [ ]:
for col in ["cell", "date", "pm25_value", "pm25_predicted", "residual"]:
    evaluation_summary[col] = validation_pd.loc[evaluation_summary["orig_index"], col].values

evaluation_summary["abs_error"] = evaluation_summary["residual"].abs()
evaluation_summary = evaluation_summary.sort_values("abs_error", ascending=True).reset_index(drop=True)

print(f"Sampled rows available for SHAP force plots: {len(evaluation_summary)}")

In [ ]:
shap.plots.beeswarm(shap_values, max_display=25)

In [ ]:
shap.plots.bar(shap_values, max_display=20)

## US EPA AQI traffic-light evaluation

In [ ]:
def pm25_to_category(x):
    if x <= 12:
        return "Good"
    elif x <= 35.4:
        return "Moderate"
    elif x <= 55.4:
        return "Unhealthy for some"
    elif x <= 150.4:
        return "Unhealthy"
    elif x > 150.5:
        return "Very Unhealthy"
    else:
        return "Unknown"

In [ ]:
def pm25_to_category_preds(x):
    if x <= 12:
        return "Good"
    elif x <= 35.4:
        return "Moderate"
    elif x <= 45.4:
        return "Unhealthy for some"
    elif x <= 100:
        return "Unhealthy"
    elif x > 100.01:
        return "Very Unhealthy"
    else:
        return "Unknown"

In [ ]:
import plotly.io as pio
from sklearn.metrics import confusion_matrix
pio.templates["custom"] = pio.templates["plotly"]
pio.templates["custom"].layout.font.family = "Roboto, sans-serif"
pio.templates.default = "custom"

In [ ]:
label_true = [pm25_to_category(x) for x in true]
label_pred = [pm25_to_category_preds(x) for x in preds]

In [ ]:
labels = ["Good", "Moderate", "Unhealthy for some", "Unhealthy", "Very Unhealthy"]
color_discrete_map = {
    "Good": "#00e400",
    "Moderate": "#ffff00",
    "Unhealthy for some": "#ff7e00",
    "Unhealthy": "#ff0000",
    "Very Unhealthy": "#8f3f97",
}

cm = confusion_matrix(label_true, label_pred, labels=labels)
cm_percent = (cm.astype(float) / cm.sum(axis=1, keepdims=True)) * 100


In [ ]:

df_cm = pd.DataFrame(cm_percent, index=labels, columns=labels)


In [ ]:
df_cm.drop("Unhealthy for some", axis=0, inplace=True)

df_cm.drop("Unhealthy for some", axis=1, inplace=True)

In [ ]:
labels.remove("Unhealthy for some")

In [ ]:

df_accuracy = pd.DataFrame(
    {
        "Good": [63.2, 35.5, 1.3],
        "Moderate": [70.4, 22.95, 6.65],
        "Unhealthy": [63.5, 23.3, 13.2],
        "Very Unhealthy": [61, 32.7, 6.3],
    },
    index=[
        "Correct",
        "One bucket error",
        "Greater than 1 bucket error",
    ]
)


In [ ]:
px.bar(df_accuracy.transpose(), orientation='v')

In [ ]:
# Transpose and melt for plotting with explicit colors
df_plot = df_accuracy.transpose().reset_index().rename(columns={"index": "AQI_Category"})
df_melt = df_plot.melt(id_vars="AQI_Category", var_name="ErrorType", value_name="Percent")

error_color_map = {
    "Correct": "#00a000",                     # green
    "One bucket error": "#90ee90",            # light green
    "Greater than 1 bucket error": "#ff0000", # red
}

fig = px.bar(
    df_melt,
    x="AQI_Category",
    y="Percent",
    color="ErrorType",
    color_discrete_map=error_color_map,
    orientation="v",
    labels={"AQI_Category": "AQI Category", "Percent": "Accuracy (%)"},
    title="Accuracy by AQI Category",
)
fig.update_layout(height=500, xaxis_tickangle=-45)
fig

In [ ]:
# Transpose and melt for plotting with explicit colors and show percent text inside bars
df_plot = df_accuracy.transpose().reset_index().rename(columns={"index": "AQI_Category"})
df_melt = df_plot.melt(id_vars="AQI_Category", var_name="ErrorType", value_name="Percent")

fig = px.bar(
    df_melt,
    x="AQI_Category",
    y="Percent",
    color="ErrorType",
    color_discrete_map=error_color_map,
    orientation="v",
    labels={"AQI_Category": "AQI Category", "Percent": "Accuracy (%)"},
    title="Accuracy by AQI Category",
    text="Percent",
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="inside")
fig.update_layout(height=500, xaxis_tickangle=-45, yaxis=dict(range=[0,100], title="Accuracy (%)"))
fig

In [ ]:
fig = px.imshow(
    df_cm,
    text_auto=".1f",  # show one decimal place (e.g. 90.0)
    labels={"x": "", "y": "", "color": "Accuracy (%)"},
    x=list(range(len(labels))),
    y=list(range(len(labels))),
    color_continuous_scale="Reds",
)

fig.update_layout(width=1200, height=1200, margin=dict(l=250, r=50, t=200, b=150))

fig.update_xaxes(side="top", tickvals=[], showticklabels=False)
fig.update_yaxes(tickvals=[], showticklabels=False)

for i, label in enumerate(labels):
    fig.add_annotation(
        x=i,
        y=1.0,
        xref="x",
        yref="paper",
        text=label,
        showarrow=False,
        font=dict(color=color_discrete_map[label], size=14),
        xanchor="center",
        yanchor="bottom",
    )

# Add custom annotations for actual (row) labels.
for i, label in enumerate(labels):
    fig.add_annotation(
        x=-0.01,
        y=i,
        xref="paper",
        yref="y",
        text=label,
        showarrow=False,
        font=dict(color=color_discrete_map[label], size=14),
        xanchor="right",
        yanchor="middle",
    )
fig.show()

## Takeaways
On this baseline model, we're underpredicting the highest PM2.5 categories because they're rare in the training data (and some sensor error may be present). Further work could focus on reweighting, augmentation, or post-processing adjustments.

In [ ]:
start_date = datetime.datetime(2025, 1, 1)
end_date = datetime.datetime(2025, 3, 1)

In [ ]:
viz_df = pl.scan_parquet(silver_path, cast_options=pl.ScanCastOptions(float_cast="upcast")).filter(pl.col("date") >= start_date)

In [ ]:
viz_preds = model.predict(viz_df.collect(engine='streaming'))
